In [1]:
from google.colab import drive
drive.mount('/content/drive/')

# Cambia la directory di lavoro in quella del tuo progetto
# (Assicurati che il percorso sia esattamente quello in cui tieni la cartella eomt sul tuo Drive)
%cd /content/drive/MyDrive/project/semantic-segmentation-roads/eomt

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).
/content/drive/MyDrive/project/semantic-segmentation-roads/eomt


In [3]:
!pip install -r requirements.txt

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 7.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 5.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 5.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 219.4/219.4 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 98.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 107.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.0/819.0 kB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 116.2 MB/

In [2]:
import os, sys, importlib

PROJECT_DIR = "/content/drive/MyDrive/project/semantic-segmentation-roads/eomt"

os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
importlib.invalidate_caches()

from my_eval import *

IMG_SIZE = (640,640)

# Assicuriamoci di essere sulla GPU
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# --- 1. CONFIGURAZIONI ---
# Usiamo la configurazione di Cityscapes per l'architettura (così avrà 19 classi)
config = get_config()

# --- 2. INIZIALIZZAZIONE DEL DATASET (Cityscapes) ---
data = get_data(config, batch_size=2, img_size=IMG_SIZE)


state_dict_path_coco = "/content/drive/MyDrive/project/eomt_coco.bin"

# --- 3. INIZIALIZZAZIONE DELL'ARCHITETTURA (Cityscapes) ---
model = get_model(config, img_size=IMG_SIZE, num_classes=data.num_classes, masked_attn_enabled=True)

# --- 4. CARICAMENTO DEI PESI DA COCO ---
model = load_weights(model, state_dict_path_coco, device).to(device)

print("\n✅ Modello e Dati pronti per il fine-tuning!")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:209: Attribute 'network' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['network'])`.


Ignored network.q.weight (shape mismatch or not existing)
Ignored network.class_head.weight (shape mismatch or not existing)
Ignored network.class_head.bias (shape mismatch or not existing)
Ignored criterion.empty_weight (shape mismatch or not existing)
Missing keys: ['network.q.weight', 'network.class_head.weight', 'network.class_head.bias', 'criterion.empty_weight']
Unexpected keys: []

✅ Modello e Dati pronti per il fine-tuning!


Spiegazione dei passaggi chiave:
1. **Configurazione Cityscapes (config_cityscapes_path)**: Inizializziamo tutta la
struttura leggendo il file di configurazione di Cityscapes. Questo assicura che il datamodule sappia che le immagini sono quelle di guida e che il network sappia che deve predire esattamente 19 classi.
2. Il trucco di **strict=False**: È il cuore di questo step. Il file eomt_coco.bin contiene i pesi di una rete addestrata a riconoscere 133 oggetti (cani, gatti, divani). Il layer finale di classificazione si aspetta di sputare fuori 133 probabilità. La nostra nuova rete, invece, ne aspetta 19. Usando model.load_state_dict(..., strict=False), diciamo a PyTorch: "Carica tutti i pesi che si incastrano perfettamente (l'estrattore di feature, l'attenzione, i transformer). Se trovi un layer che ha dimensioni diverse, saltalo".
3. **Risultato finale**: Abbiamo un estrattore di caratteristiche "intelligente" (vede benissimo forme e contorni perché ha studiato su COCO), ma un classificatore finale "vuoto e ignorante", pronto per imparare a mappare quelle forme sulle 19 classi stradali di Cityscapes.

Step 2: congelamento pesi

 La rete principale si chiama EoMT e ha questi componenti chiave:
*   encoder (il "cervello" o backbone, solitamente un grosso Transformer - ViT)
*   q (le "query" della rete)
*   mask_head (la parte che disegna le maschere per la segmentazione)
*   class_head (il layer finale che dice "questa maschera è un pedone", ed è quello che produce le 19 probabilità per Cityscapes).

Poiché nello Step 1 abbiamo ignorato i vecchi pesi della class_head (che erano per COCO), la nostra nuova class_head ha **pesi casuali**. Se provassimo ad addestrare tutta la rete insieme da subito, l'errore enorme causato dalla testa casuale andrebbe a "sporcare" (tramite backpropagation) l'ottimo lavoro che l'encoder ha imparato su COCO.

Ecco perché il documento ti dice: "finetune just the prediction head". Vogliamo che la nuova testa impari a leggere l'encoder perfetto.

La class_head ha pesi con valori completamente casuali (random).
Perché:


1.   La Nascita (Pesi Casuali): Quando nello Step 1 hai eseguito il comando per creare la rete (network = network_cls(...)), PyTorch ha costruito la tua architettura e ha assegnato a tutti i layer della rete dei numeri puramente casuali (seguendo una distribuzione matematica standard per inizializzare le reti neurali). In quel preciso istante, la tua rete è "neonata": non sa riconoscere nulla e produrrebbe solo rumore.
2.   L'Iniezione di Conoscenza (Il load_state_dict): Subito dopo, abbiamo caricato il file eomt_coco.bin. Questo file contiene i pesi "intelligenti" che i creatori del modello hanno ottenuto dopo aver fatto allenare la rete per giorni sui server.
3. Il filtro dello strict=False: La funzione load_state_dict ha iniziato a sovrascrivere i numeri casuali del tuo modello neonato con i numeri intelligenti di COCO, layer per layer. Ma quando è arrivata a model.network.class_head, si è accorta di un problema: "Il file di COCO mi dice che qui c'è una matrice per 133 classi. Ma la rete che ho davanti ha spazio solo per 19 classi!"
Avendo messo strict=False, la funzione ha fatto spallucce, ha ignorato i pesi di COCO per quel layer specifico ed è andata oltre.

Quindi, qual è lo stato della rete prima del Training?
*   Il Backbone (Encoder) e il Transformer: Hanno i valori pre-addestrati di COCO. Sanno riconoscere perfettamente i contorni di un'auto, le strisce pedonali o i bordi dei palazzi.
*   La Prediction Head (class_head): Ha mantenuto i valori casuali che PyTorch le aveva assegnato all'inizio.

Perché questo è fondamentale?
Poiché la testa ha valori casuali, se provassi a far fare una predizione al modello in questo esatto momento, otterresti un disastro: l'encoder estrarrebbe forme perfette (es. "ecco un pedone"), ma la testa, avendo pesi random, potrebbe mappare quel pedone alla classe "Cielo" o "Semaforo" in modo del tutto arbitrario.

Ecco perché il documento ti suggerisce di addestrare inizialmente SOLO la prediction head. Avendo l'encoder congelato, l'addestramento si concentrerà esclusivamente sul modificare quei "numeri casuali" della testa, trasformandoli nei numeri corretti per collegare "forma del pedone" -> "classe Pedone di Cityscapes".

In [3]:
# --- 1. CONGELAMENTO GLOBALE (FREEZING) ---
# Diciamo a PyTorch di non calcolare i gradienti per NESSUN parametro del modello
for param in model.parameters():
    param.requires_grad = False

# --- 2. SCONGELAMENTO DELLA PREDICTION HEAD ---
# Riattiviamo i gradienti SOLO per il layer di classificazione finale
# (Guardando nel codice originale, si chiama esattamente "class_head")
for param in model.network.class_head.parameters():
    param.requires_grad = True

# --- 3. (OPZIONALE) SCONGELARE LE QUERY ---
# Nelle architetture a maschere (come Mask2Former/EoMT), a volte aiuta
# scongelare anche le query. Inizia con solo class_head, ma se non impara,
# puoi decommentare queste due righe per scongelare anche le query:
# for param in model.network.q.parameters():
#     param.requires_grad = True

# --- 4. VERIFICA ---
# Stampiamo un riepilogo per essere sicuri di non far esplodere la RAM di Colab!
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print(f"Parametri TOTALI del modello: {total_params:,}")
print(f"Parametri ADDESTRABILI (scongelati): {trainable_params:,}")
print(f"Percentuale di pesi in addestramento: {(trainable_params/total_params)*100:.4f}%")


Parametri TOTALI del modello: 93,498,644
Parametri ADDESTRABILI (scongelati): 15,380
Percentuale di pesi in addestramento: 0.0164%


Step 3: configurare il Trainer e avviare il Training

In [5]:
import os
from datetime import datetime
import torch

max_epochs = 10
real_epochs = 10

# -------------------------
# 0. Nome run e cartelle
# -------------------------
run_name = datetime.now().strftime(f"head_only_resume_to_{max_epochs}ep_%Y%m%d_%H%M")

ckpt_dir = f"/content/drive/MyDrive/project/checkpoints/{run_name}"
os.makedirs(ckpt_dir, exist_ok=True)

old_ckpt_dir = "/content/drive/MyDrive/project/checkpoints/head_only_5ep_20260508_0735"
resume_ckpt_path = f"{old_ckpt_dir}/last.ckpt"
print("Checkpoint exists:", os.path.exists(resume_ckpt_path))
print(resume_ckpt_path)



Checkpoint exists: True
/content/drive/MyDrive/project/checkpoints/head_only_5ep_20260508_0735/last.ckpt


In [ ]:
# import os
# from datetime import datetime
# import torch

# max_epochs = 10
# real_epochs = 5

# # -------------------------
# # 0. Nome run e cartelle
# # -------------------------
# run_name = datetime.now().strftime(f"head_only_resume_to_{max_epochs}ep_%Y%m%d_%H%M")

# ckpt_dir = f"/content/drive/MyDrive/project/checkpoints/{run_name}"
# os.makedirs(ckpt_dir, exist_ok=True)

# old_ckpt_dir = "/content/drive/MyDrive/project/checkpoints/head_only_5ep_20260508_0735"
# resume_ckpt_path = f"{old_ckpt_dir}/last.ckpt"
# print("Checkpoint exists:", os.path.exists(resume_ckpt_path))
# print(resume_ckpt_path)

# ckpt = torch.load(resume_ckpt_path, map_location=device, weights_only=False)
# missing, unexpected = model.load_state_dict(ckpt["state_dict"], strict=False)

# print("Missing keys:", len(missing))
# print("Unexpected keys:", len(unexpected))


Checkpoint exists: True
/content/drive/MyDrive/project/checkpoints/head_only_5ep_20260508_0735/last.ckpt
Missing keys: 0
Unexpected keys: 0


In [ ]:
# import wandb
# import lightning.pytorch as pl
# from lightning.pytorch.callbacks import ModelCheckpoint, LearningRateMonitor
# from lightning.pytorch.loggers import WandbLogger



# # 1. Eseguiamo il login a Weights & Biases
# # (Ti chiederà la chiave API se non sei già loggata nella sessione corrente)
# wandb.login()

# # 2. Creiamo il logger: tutto verrà salvato sul tuo profilo nel progetto "eomt-cityscapes"
# wandb_logger = WandbLogger(project="eomt-cityscapes", name=run_name, log_model=False)

# # Log config utile su WandB
# wandb_logger.experiment.config.update({
#     "run_name": run_name,
#     "max_epochs": max_epochs,
#     "real_epochs": real_epochs,
#     "precision": "16-mixed",
#     "batch_size": getattr(data, "batch_size", None),
#     "img_size": getattr(data, "img_size", None),
#     "num_classes": getattr(data, "num_classes", None),
#     "finetuning": "class_head_only_resume_to_10ep",
#     "resume_from": resume_ckpt_path
# })

# # 3. Salvataggio dei pesi (checkpoint) su Drive
# checkpoint_callback = ModelCheckpoint(
#     dirpath=ckpt_dir,
#     filename="best-{epoch:02d}",
#     save_last=True,
#     save_top_k=2,
#     monitor="metrics/val_iou_all",
#     mode="max"
# )

# # Log learning rate su WandB
# lr_monitor = LearningRateMonitor(logging_interval="step")

# print(f"Checkpoint directory: {ckpt_dir}")



# print("Configurazione del PyTorch Lightning Trainer...")

# trainer = pl.Trainer(
#     max_epochs=real_epochs,
#     accelerator="gpu",
#     devices=1,
#     precision="16-mixed",
#     logger=wandb_logger,
#     callbacks=[checkpoint_callback, lr_monitor],

#     log_every_n_steps=10,

#     # Per run seria: sanity check attivo, ma non enorme
#     num_sanity_val_steps=2,

#     # Utile su Colab per non perdere tutto se crasha
#     enable_checkpointing=True,

#     # Se vuoi evitare training infinito per bug strani
#     enable_progress_bar=True,

#     # Ricordati: se vedi che si incastra sul Sanity Checking, ferma tutto,
#     # copia i file ZIP in locale su /content/ e togli questi limiti!
#     # limit_train_batches=20,
#     # limit_val_batches=5,
# )

# print("🚀 Avvio del Fine-Tuning!")
# trainer.fit(model, datamodule=data)

# # Alla fine dell'addestramento, diciamo a WandB che abbiamo finito
# wandb.finish()


INFO: Using 16bit Automatic Mixed Precision (AMP)
INFO:lightning.pytorch.utilities.rank_zero:Using 16bit Automatic Mixed Precision (AMP)
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: HPU available: False, using: 0 HPUs
INFO:lightning.pytorch.utilities.rank_zero:HPU available: False, using: 0 HPUs


Checkpoint directory: /content/drive/MyDrive/project/checkpoints/head_only_resume_to_10ep_20260510_1055
Configurazione del PyTorch Lightning Trainer...
🚀 Avvio del Fine-Tuning!


INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO: Loading `train_dataloader` to estimate number of stepping batches.
INFO:lightning.pytorch.utilities.rank_zero:Loading `train_dataloader` to estimate number of stepping batches.
INFO: 
  | Name      | Type                   | Params | Mode 
-------------------------------------------------------------
0 | network   | EoMT                   | 93.5 M | train
1 | criterion | MaskClassificationLoss | 0      | train
2 | metrics   | ModuleList             | 0      | train
-------------------------------------------------------------
15.4 K    Trainable params
93.5 M    Non-trainable params
93.5 M    Total params
373.995   Total estimated model params size (MB)
292       Modules in train mode
0         Modules in eval mode
INFO:lightning.pytorch.callbacks.model_summary:
  | Name      | Type                   | Params | Mode 
----------------------------------

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Una volta che l'addestramento (anche di sole 5 epoche) è terminato, il progetto ti chiede di:

Valutare il modello che hai appena addestrato usando la metrica mIoU.
Confrontarlo con gli altri modelli.
Per fare la valutazione, ti basterà ricopiare esattamente la cella che valuta la mIoU che era presente nel tuo Step 4 originario, ma passandogli questo nuovo model che ora è stato perfezionato su Cityscapes!

In [6]:
import wandb
import lightning.pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint, LearningRateMonitor
from lightning.pytorch.loggers import WandbLogger



# 1. Eseguiamo il login a Weights & Biases
# (Ti chiederà la chiave API se non sei già loggata nella sessione corrente)
wandb.login()

# 2. Creiamo il logger: tutto verrà salvato sul tuo profilo nel progetto "eomt-cityscapes"
wandb_logger = WandbLogger(project="eomt-cityscapes", name=run_name, log_model=False)

# Log config utile su WandB
wandb_logger.experiment.config.update({
    "run_name": run_name,
    "max_epochs": max_epochs,
    "real_epochs": real_epochs,
    "precision": "16-mixed",
    "batch_size": getattr(data, "batch_size", None),
    "img_size": getattr(data, "img_size", None),
    "num_classes": getattr(data, "num_classes", None),
    "finetuning": "class_head_only_resume_to_10ep",
    "resume_from": resume_ckpt_path
})

# 3. Salvataggio dei pesi (checkpoint) su Drive
checkpoint_callback = ModelCheckpoint(
    dirpath=ckpt_dir,
    filename="best-{epoch:02d}",
    save_last=True,
    save_top_k=2,
    monitor="metrics/val_iou_all",
    mode="max"
)

# Log learning rate su WandB
lr_monitor = LearningRateMonitor(logging_interval="step")

print(f"Checkpoint directory: {ckpt_dir}")



print("Configurazione del PyTorch Lightning Trainer...")

trainer = pl.Trainer(
    max_epochs=real_epochs,
    accelerator="gpu",
    devices=1,
    precision="16-mixed",
    logger=wandb_logger,
    callbacks=[checkpoint_callback, lr_monitor],

    log_every_n_steps=10,

    # Per run seria: sanity check attivo, ma non enorme
    num_sanity_val_steps=2,

    # Utile su Colab per non perdere tutto se crasha
    enable_checkpointing=True,

    # Se vuoi evitare training infinito per bug strani
    enable_progress_bar=True,

    # Ricordati: se vedi che si incastra sul Sanity Checking, ferma tutto,
    # copia i file ZIP in locale su /content/ e togli questi limiti!
    # limit_train_batches=20,
    # limit_val_batches=5,
)

print("🚀 Avvio del Fine-Tuning!")
trainer.fit(model, datamodule=data, ckpt_path=resume_ckpt_path)

# Alla fine dell'addestramento, diciamo a WandB che abbiamo finito
wandb.finish()


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ai-group-3434 (ai-group-3434-politecnico-di-torino) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.


...

Checkpoint directory: /content/drive/MyDrive/project/checkpoints/head_only_resume_to_10ep_20260511_0822
Configurazione del PyTorch Lightning Trainer...
🚀 Avvio del Fine-Tuning!


INFO: Restoring states from the checkpoint path at /content/drive/MyDrive/project/checkpoints/head_only_5ep_20260508_0735/last.ckpt
INFO:lightning.pytorch.utilities.rank_zero:Restoring states from the checkpoint path at /content/drive/MyDrive/project/checkpoints/head_only_5ep_20260508_0735/last.ckpt
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/callbacks/model_checkpoint.py:362: The dirpath has changed from '/content/drive/.shortcut-targets-by-id/1kGi4cSNJjM14ClVvPXJ9VY70JDkofHGn/project/checkpoints/head_only_5ep_20260508_0735' to '/content/drive/MyDrive/project/checkpoints/head_only_resume_to_10ep_20260511_0822', therefore `best_model_score`, `kth_best_model_path`, `kth_value`, `last_model_path` and `best_k_models` won't be reloaded. Only `best_model_path` will be reloaded.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO: Loading `train_dataloader` to estimate number of stepping batches

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

RuntimeError: result type ComplexFloat can't be cast to the desired output type Float